In [9]:
"""Q1: Simple NLP preprocessing pipeline (tokenization, stopwords, stemming)."""
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize

COMMON_STOPWORDS = {
    "a", "an", "the", "and", "are", "is", "am", "was", "were",
    "in", "on", "at", "to", "for", "of", "by", "with", "from"
}


def preprocess(sentence):

    # preserve_line=True avoids requiring the separate punkt tokenizer data.
    tokens = word_tokenize(sentence, preserve_line=True)

    try:
        stop_words = set(stopwords.words("english"))
    except LookupError:
        stop_words = COMMON_STOPWORDS

    # Remove stopwords and punctuation, keeping word order.
    filtered = [word for word in tokens
                if word.isalpha() and word.lower() not in stop_words]

    stemmer = PorterStemmer()
    stemmed = [stemmer.stem(word.lower()) for word in filtered]

    print("Original Tokens:", tokens)
    print("Tokens Without Stopwords:", filtered)
    print("Stemmed Words:", stemmed)
    return tokens, filtered, stemmed


sentence = "NLP techniques are used in virtual assistants like Alexa and Siri."
preprocess(sentence);

Original Tokens: ['NLP', 'techniques', 'are', 'used', 'in', 'virtual', 'assistants', 'like', 'Alexa', 'and', 'Siri', '.']
Tokens Without Stopwords: ['NLP', 'techniques', 'used', 'virtual', 'assistants', 'like', 'Alexa', 'Siri']
Stemmed Words: ['nlp', 'techniqu', 'use', 'virtual', 'assist', 'like', 'alexa', 'siri']


In [10]:
"""Q2: Named entity recognition using spaCy."""
import spacy


def show_entities(sentence):

    try:
        nlp = spacy.load("en_core_web_sm")
    except OSError as exc:
        raise SystemExit(
            "Install the English model first: python -m spacy download en_core_web_sm"
        ) from exc

    doc = nlp(sentence)
    for ent in doc.ents:
        # end_char is exclusive: sentence[start_char:end_char] == ent.text.
        print(f"{ent.text} | {ent.label_} | Start: {ent.start_char} | End: {ent.end_char}")
    return [(ent.text, ent.label_, ent.start_char, ent.end_char)
            for ent in doc.ents]


# The leading space is intentionally preserved from the assignment.
sentence = (" LeBron James as the NBA player of the United States in "
            "Laker and won the NBA Final MVP in 2020.")
show_entities(sentence)

LeBron James | PERSON | Start: 1 | End: 13
NBA | ORG | Start: 21 | End: 24
the United States | GPE | Start: 35 | End: 52
Laker | GPE | Start: 56 | End: 61
MVP | ORG | Start: 84 | End: 87
2020 | DATE | Start: 91 | End: 95


[('LeBron James', 'PERSON', 1, 13),
 ('NBA', 'ORG', 21, 24),
 ('the United States', 'GPE', 35, 52),
 ('Laker', 'GPE', 56, 61),
 ('MVP', 'ORG', 84, 87),
 ('2020', 'DATE', 91, 95)]

In [11]:
"""Q3: Scaled dot-product attention using NumPy."""
import numpy as np


def scaled_dot_product_attention(Q, K, V):

    # 1. Compute the query-key similarities.
    scores = np.matmul(Q, K.T)

    # 2. Scale by the square root of the key vector dimension.
    d = K.shape[-1]
    scaled_scores = scores / np.sqrt(d)

    # 3. Apply a stable softmax across the keys (each row sums to 1).
    exp_scores = np.exp(scaled_scores - np.max(scaled_scores, axis=-1, keepdims=True))
    weights = exp_scores / np.sum(exp_scores, axis=-1, keepdims=True)

    # 4. Take the weighted sum of value vectors.
    output = np.matmul(weights, V)
    return weights, output


Q = np.array([[1, 0, 1, 0], [0, 1, 0, 1]])
K = np.array([[1, 0, 1, 0], [0, 1, 0, 1]])
V = np.array([[1, 2, 3, 4], [5, 6, 7, 8]])

weights, output = scaled_dot_product_attention(Q, K, V)
print("Attention Weights:\n", np.round(weights, 3))
print("Final Output Matrix:\n", np.round(output, 3))

Attention Weights:
 [[0.731 0.269]
 [0.269 0.731]]
Final Output Matrix:
 [[2.076 3.076 4.076 5.076]
 [3.924 4.924 5.924 6.924]]
